# Iterative Binding Optimization

**BioPipelines example** — iterative directed evolution loop that uses LigandMPNN to propose sequence variants and Boltz2 to evaluate their binding affinity, keeping only the best candidate each cycle.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e .
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba
!micromamba create -f environments/biopipelines.yaml -y

Cloning into 'biopipelines'...
remote: Enumerating objects: 7575, done.
remote: Counting objects: 100% (945/945), done.
remote: Compressing objects: 100% (347/347), done.
remote: Total 7575 (delta 647), reused 781 (delta 587), pack-reused 6630 (from 1)
Receiving objects: 100% (7575/7575), 16.67 MiB | 19.25 MiB/s, done.
Resolving deltas: 100% (5802/5802), done.
/content/biopipelines
Obtaining file:///content/biopipelines
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 15.0 MB/s eta 0:00:00
  Building editable for biopipelines (pyproject.toml) ... done
  Created wheel for biopipelines: filename=biopipelines-1.1.1-0.edi

In [2]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.boltz2 import Boltz2
from biopipelines.ligand_mpnn import LigandMPNN
from biopipelines.mutation_profiler import MutationProfiler

with Pipeline(project="Setup", job="InstallTools"):
    Boltz2.install()
    LigandMPNN.install()
    MutationProfiler.install()


Running Boltz2_installation (step 1)
=== Activating Environment ===
Requested: biopipelines
Environment: biopipelines
Location: /root/.local/share/mamba/envs/biopipelines
Python: /root/.local/share/mamba/envs/biopipelines/bin/python
Python version: Python 3.12.13
=== Installing Boltz2 ===
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                           ⧖ Starting
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                              

## Cell 3: Iterative Binding Optimization Pipeline

Starting from the NocT protein (PDB: 5OT9) and Histopine as ligand, this pipeline runs 5 cycles of:
1. Identify binding-pocket residues within 5 Å of the ligand
2. Generate 1000 sequence variants with LigandMPNN
3. Build a mutation frequency profile
4. Compose 3 candidate sequences by weighted random sampling (max 3 mutations)
5. Predict binding affinities with Boltz2 and keep the best candidate

In [3]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.boltz2 import Boltz2
from biopipelines.ligand_mpnn import LigandMPNN
from biopipelines.distance_selector import DistanceSelector
from biopipelines.mutation_profiler import MutationProfiler
from biopipelines.mutation_composer import MutationComposer
from biopipelines.panda import Panda

with Pipeline(project="NocT", job=f"IterativeBindingOptimization"):
    Resources(gpu="A100", time="24:00:00", memory="16GB")
    protein = Sequence("5OT9")
    ligand = Ligand("Histopine")
    original = Boltz2(proteins=protein,
                      ligands=ligand)

    current_best = Panda(tables=original.tables.affinity,
                         pool=original)
    for cycle in range(5):
        Suffix(f"Cycle{cycle+1}")
        pocket = DistanceSelector(structures=current_best,
                                  ligand="LIG",
                                  distance=5)
        variants = LigandMPNN(structures=current_best,
                            ligand="LIG",
                            num_sequences=50,
                            num_batches=20,
                            redesigned=pocket.tables.selections.within)
        profile = MutationProfiler(original=current_best,
                                   mutants=variants)
        candidates = MutationComposer(frequencies=profile.tables.absolute_frequencies,
                                      num_sequences=3,
                                      mode="weighted_random",
                                      max_mutations=3)
        predicted = Boltz2(proteins=candidates,
                           ligands=ligand)
        current_best = Panda(tables=[current_best.tables.result,
                                    predicted.tables.affinity],
                            operations=[Panda.concat(),
                                        Panda.sort("affinity_probability_binary",
                                                   ascending=False),
                                        Panda.head(1)],
                            pool=[current_best,
                                  predicted])
current_best

Streaming output truncated to the last 5000 lines.
Checking input data.
Processing 1 inputs with 1 threads.

  0%|          | 0/1 [00:00<?, ?it/s]Generating MSA for /content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/003_Boltz2/_configuration/config_files/5OT9+Histopine.yaml with 1 protein entities.
Calling MSA server for target 5OT9+Histopine with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


  0%|          | 0/150 [elapsed: 00:00 remaining: ?]

SUBMIT:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]

COMPLETE:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:01 remaining: 00:00]

100%|██████████| 1/1 [00:01<00:00,  1.95s/it]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/root/.local/share/mamba/envs/Boltz2Env/lib/pyt

StandardizedOutput({'structures': DataStream(name='structures', format='pdb', items=1, files=1, map_table=set), 'sequences': DataStream(name='sequences', format='csv', items=1, files=0, map_table=set), 'compounds': DataStream(name='compounds', format='csv', items=1, files=1, map_table=set), 'msas': DataStream(name='msas', format='csv', items=1, files=1, map_table=set), 'rendering_parameters': {'structures': {'color_by': 'plddt', 'plddt_upper': 100}}, 'tables': {'result': TableInfo(name='result', path='/content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/034_Panda_Cycle5/tables/concat_sort_head.csv', columns=['id', 'input_file', 'affinity_pred_value', 'affinity_probability_binary'], count=variable), 'missing': TableInfo(name='missing', path='/content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/034_Panda_Cycle5/tables/missing.csv', columns=['id', 'removed_by', 'cause'], count=variable), 'structures': TableInfo(name='structures', path='/content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/034_Panda_Cycle5/structures/structures_map.csv', columns=['id', 'file'], count=1), 'confidence': TableInfo(name='confidence', path='/content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/034_Panda_Cycle5/tables/confidence.csv', columns=['id', 'input_file', 'confidence_score', 'ptm', 'iptm', 'complex_plddt', 'complex_iplddt'], count=1), 'sequences': TableInfo(name='sequences', path='/content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/034_Panda_Cycle5/sequences/sequences.csv', columns=['id', 'sequence'], count=1), 'msas': TableInfo(name='msas', path='/content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/034_Panda_Cycle5/msas/msas_map.csv', columns=['id', 'sequences.id', 'sequence', 'msa_file'], count=1), 'affinity': TableInfo(name='affinity', path='/content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/034_Panda_Cycle5/tables/affinity.csv', columns=['id', 'input_file', 'affinity_pred_value', 'affinity_probability_binary'], count=1), 'compounds': TableInfo(name='compounds', path='/content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/034_Panda_Cycle5/compounds/compounds.csv', columns=['id', 'format', 'smiles', 'ccd'], count=1)}, 'output_folder': '/content/biopipelines/outputs/NocT/IterativeBindingOptimization_001/034_Panda_Cycle5'})